# Heads Up - Data Preprocessing

**Input:** raw DataCo Smart Supply Chain data (180,519 rows, order line level)
**Output:** `train.csv`, `val.csv`, `test.csv` - clean, aggregated, order level datasets ready for feature engineering (Stage 4) and modelling (Stage 5/6)

---

## Purpose of this notebook

This is the consolidated Stage 3 deliverable. It takes the raw, order line level data and the findings from Stage 2 (EDA) and works through five sequential steps to produce modelling ready train/validation/test sets:

1. **Aggregation** - collapse line items to one row per order
2. **Missing values** - resolve `Order Zipcode`
3. **Categorical standardization** - resolve the `Order Country` language question and the `Category Id`/`Category Name` mismatch
4. **Outlier handling** - cap `Benefit per order`, define the secondary lens's value field
5. **Chronological train/validation/test split** - for modelling and Optuna based tuning

Each section states what we found, what we decided, and why - several steps here diverge from what a naive default approach would do, because checking the data directly surfaced real issues worth catching before they reached modelling.


## Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')

RAW_DATA_PATH = Path('../data/raw data/DataCoSupplyChainDataset.csv') 
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

try:
    df = pd.read_csv(RAW_DATA_PATH, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(RAW_DATA_PATH, encoding='ISO-8859-1')

print(f"Raw shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique orders: {df['Order Id'].nunique():,}")


Raw shape: 180,519 rows x 53 columns
Unique orders: 65,752


## 1. Aggregation - order line to order level

EDA confirmed the raw data is at order line level (2.75 lines/order average). Before applying any aggregation rule, we check which columns are actually constant within an order versus which genuinely vary - this matters because a careless `first` on a varying column silently discards information.


In [3]:
candidate_columns = [
    'Shipping Mode', 'Order Region', 'Order Country', 'Order City', 'Order State',
    'Customer Segment', 'Type', 'Days for shipment (scheduled)',
    'Category Name', 'Category Id', 'Late_delivery_risk', 'Delivery Status',
]
candidate_columns = [c for c in candidate_columns if c in df.columns]

variability_check = df.groupby('Order Id')[candidate_columns].nunique().max()
variability_check.sort_values(ascending=False)


Category Name                    5
Category Id                      5
Shipping Mode                    1
Order Region                     1
Order City                       1
Order Country                    1
Order State                      1
Customer Segment                 1
Days for shipment (scheduled)    1
Type                             1
Late_delivery_risk               1
Delivery Status                  1
dtype: int64

**What we found:** every planned field is genuinely constant within an order - **except `Category Name` and `Category Id`, which vary up to 5 times within a single order.** This makes sense: an order can span multiple product categories (e.g. camping gear plus a water sports item), while `Shipping Mode`, `Order Region`, etc. are true order-level properties (one shipment, one destination).

**This means a naive `first` aggregation for category would silently keep only one of up to 5 categories, arbitrarily and inconsistently.** We use the mode (most frequent category) instead, and add a new feature - `n_distinct_categories` - to preserve the real "this was a mixed-category order" signal rather than discarding it.


In [4]:
def mode_or_first(series):
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else series.iloc[0]

agg_rules = {
    'Late_delivery_risk': 'first',
    'Delivery Status': 'first',
    'Shipping Mode': 'first',
    'Order Region': 'first',
    'Order Country': 'first',
    'Order City': 'first',
    'Order State': 'first',
    'Order Zipcode': 'first',
    'Customer Segment': 'first',
    'Type': 'first',
    'Days for shipment (scheduled)': 'first',
    'Category Name': mode_or_first,
    'Category Id': mode_or_first,
    'Sales': 'sum',
    'Order Item Quantity': 'sum',
    'Benefit per order': 'sum',
    'order date (DateOrders)': 'first',
}
agg_rules = {k: v for k, v in agg_rules.items() if k in df.columns}

orders_df = df.groupby('Order Id').agg(agg_rules).reset_index()
orders_df['n_line_items'] = df.groupby('Order Id').size().values
orders_df['n_distinct_categories'] = df.groupby('Order Id')['Category Name'].nunique().values

print(f"Shape after aggregation: {orders_df.shape[0]:,} rows x {orders_df.shape[1]} columns")

multi_category_orders = (orders_df['n_distinct_categories'] > 1).sum()
print(f"Orders with more than 1 distinct category: {multi_category_orders:,} ({multi_category_orders/len(orders_df)*100:.2f}%)")

assert orders_df.shape[0] == df['Order Id'].nunique(), "Row count mismatch after aggregation!"
assert orders_df['Late_delivery_risk'].isnull().sum() == 0, "Target has missing values after aggregation!"
print(f"\nClass balance after aggregation: {orders_df['Late_delivery_risk'].value_counts(normalize=True).round(4).to_dict()}")


Shape after aggregation: 65,752 rows x 20 columns
Orders with more than 1 distinct category: 44,563 (67.77%)

Class balance after aggregation: {1: 0.5482, 0: 0.4518}


**What we found:** the scale of the multi category issue is much bigger than a rare edge case - **44,563 orders (67.77% of the entire dataset) contain more than one product category.** Had this gone uncorrected with a naive `first` aggregation, two thirds of the dataset would have carried an essentially arbitrary category label. Class balance after aggregation (54.82% late / 45.18% not late) matches the line item level EDA finding almost exactly - confirming the target collapsed correctly.

**Decision logged:** `Category Name`/`Category Id` aggregated via mode, not `first`; `n_line_items` and `n_distinct_categories` added as new order level features.


## 2. Missing values - resolving `Order Zipcode`

EDA flagged `Order Zipcode` as likely systemically missing (tied to country) rather than randomly missing. We confirm this directly before deciding drop vs. flag.


In [6]:
print(f"Order Zipcode missing: {orders_df['Order Zipcode'].isnull().sum():,} of {len(orders_df):,} orders ({orders_df['Order Zipcode'].isnull().mean()*100:.2f}%)")

zipcode_availability_by_country = orders_df.groupby('Order Country')['Order Zipcode'].apply(lambda x: x.notna().mean()).sort_values()
print("\nLowest zipcode availability countries:")
zipcode_availability_by_country.head(10)


Order Zipcode missing: 57,482 of 65,752 orders (87.42%)

Lowest zipcode availability countries:


Order Country
Afganistán      0.0
Albania         0.0
Alemania        0.0
Angola          0.0
Arabia Saudí    0.0
Argelia         0.0
Argentina       0.0
Armenia         0.0
Australia       0.0
Austria         0.0
Name: Order Zipcode, dtype: float64

**What we found:** confirmed systemic, not random. 87.42% of orders lack a zipcode, and every one of the lowest availability countries shows **exactly 0.0%** availability - a hard structural pattern (zipcodes only meaningfully apply to a small subset of countries in this dataset), not partial noise.

**Decision:** drop `Order Zipcode` entirely rather than keeping a binary flag - with the vast majority of countries at 0% availability, a flag would be near redundant with `Order Country` itself.


In [7]:
orders_df = orders_df.drop(columns=['Order Zipcode'], errors='ignore')

remaining_missing = orders_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f"Remaining missing values: {remaining_missing.to_dict() if len(remaining_missing) > 0 else 'none'}")
print(f"Shape: {orders_df.shape[0]:,} rows x {orders_df.shape[1]} columns")


Remaining missing values: none
Shape: 65,752 rows x 19 columns


**What we found:** no other missing values remain. `Customer Lname`/`Customer Zipcode` (the two negligible missing fields found in EDA) were never carried into this aggregated dataset to begin with, since they're not planned model inputs.


## 3. Categorical standardization

Two things flagged in EDA needed direct investigation here: whether `Order Country` mixes English/Spanish naming, and the `Category Id`/`Category Name` count mismatch (51 vs. 50 unique values).


In [8]:
all_countries = sorted(orders_df['Order Country'].unique())
print(f"Total unique Order Country values: {len(all_countries)}")


Total unique Order Country values: 164


**What we found - this corrects the earlier EDA assumption rather than confirming it:** looking at the complete list of 164 countries, `Order Country` is **consistently in Spanish throughout** - it is not actually a mix of English and Spanish. The EDA sample that looked "mixed" (Ghana, Canada, Portugal, Argentina, Austria appearing alongside Hungría, Irlanda, Costa de Marfil) was misleading: those "English looking" entries are simply countries whose Spanish and English spellings happen to coincide. **No language standardization fix is actually needed.** One cosmetic, non duplicating quirk: `SudAfrica` (South Africa) is written without a space, unlike other multi word names - noted but not corrected, since it doesn't create a duplicate category.


In [9]:
category_id_to_name = orders_df.groupby('Category Id')['Category Name'].nunique()
name_to_category_id = orders_df.groupby('Category Name')['Category Id'].nunique()

print(f"Category Ids mapping to more than one name: {(category_id_to_name > 1).sum()}")
print(f"Category Names mapping to more than one Id: {(name_to_category_id > 1).sum()}")


Category Ids mapping to more than one name: 28
Category Names mapping to more than one Id: 24


**What we found - much bigger than EDA's "51 vs. 50" count suggested:** this is a widespread many to many inconsistency - 28 `Category Id` values map to more than one name, and 24 names are shared across multiple IDs (e.g. `Category Id` 17 = both "Accessories" and "Cleats"). The hypothesis: category IDs are scoped **within a department**, not globally unique.


In [10]:
paired_check = df.groupby(['Department Id', 'Category Id'])['Category Name'].nunique()
still_inconsistent = paired_check[paired_check > 1]
print(f"(Department Id, Category Id) pairs still mapping to more than one name: {len(still_inconsistent)}")


(Department Id, Category Id) pairs still mapping to more than one name: 0


**What we found:** **zero** `(Department Id, Category Id)` pairs remain inconsistent - the department scoping theory is confirmed cleanly. This isn't a data quality error at all; it's a normal ID scoping convention. `Category Id` alone is not a reliable standalone identifier, but `Category Name` (already the feature carried through aggregation, correctly reflecting each line item's true department aware category) is confirmed reliable to use as is.

**Decisions logged:** no country standardization needed (EDA assumption corrected). `Category Name` confirmed as the canonical category feature; `Category Id` excluded as a standalone feature unless explicitly paired with `Department Id`.


## 4. Outlier handling

EDA found `Benefit per order` had substantial outliers at line item level (10.49% by IQR, minimum –$4,274.98). We re-check at order level, since summing line items changes the distribution, then cap for model stability and resolve how the secondary lens's priority formula should treat order value.


In [11]:
for col in ['Sales', 'Benefit per order']:
    q1, q3 = orders_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((orders_df[col] < lower) | (orders_df[col] > upper)).sum()
    print(f"{col}: {n_outliers:,} outliers ({n_outliers/len(orders_df)*100:.2f}%) outside [{lower:.2f}, {upper:.2f}]")


Sales: 152 outliers (0.23%) outside [-535.03, 1600.94]
Benefit per order: 5,188 outliers (7.89%) outside [-218.51, 384.98]


**What we found:** both outlier rates dropped after aggregation - `Sales` from 0.27% to 0.23%, and more notably `Benefit per order` from **10.49% down to 7.89%**. This is expected: summing line items smooths out extremes rather than compounding them. Note this IQR figure characterizes the *extent* of the outlier problem; the actual *treatment* below uses percentile capping, a different method that affects a different (smaller) number of rows - both numbers are correct, they're just measuring different things.


In [12]:
lower_cap = orders_df['Benefit per order'].quantile(0.01)
upper_cap = orders_df['Benefit per order'].quantile(0.99)
orders_df['Benefit_per_order_capped'] = orders_df['Benefit per order'].clip(lower=lower_cap, upper=upper_cap)

n_capped = ((orders_df['Benefit per order'] < lower_cap) | (orders_df['Benefit per order'] > upper_cap)).sum()
print(f"Capped Benefit per order to [{lower_cap:.2f}, {upper_cap:.2f}] — {n_capped:,} orders affected ({n_capped/len(orders_df)*100:.2f}%)")

# Secondary lens: use Sales (always positive) as the value term, avoiding the negative-profit sign problem entirely
orders_df['priority_value_component'] = orders_df['Sales']


Capped Benefit per order to [-645.22, 422.12] — 1,316 orders affected (2.00%)


**Decision, reasoned here:** `Sales` (always positive) drives the secondary lens's priority score, not raw `Benefit per order` - this avoids a large loss order being miscounted as "high value, deprioritise nothing." Capped `Benefit per order` remains available for reporting, but isn't the priority rank driver.


## 5. Chronological train / validation / test split

Per the EDA finding - no meaningful year over year drift, but a thin single month 2018 tail that's too small/seasonally narrow to hold out alone - we use a **proportion based chronological split**, not a random or calendar year split.

**Three way, not two way:** since Optuna based hyperparameter tuning (Stage 8) will run many trials across 4 candidate models, evaluating every trial with full cross validation is computationally expensive. A single fixed validation set lets each trial be scored once - the test set stays fully untouched until final evaluation (Stage 7).


In [13]:
orders_df['order date (DateOrders)'] = pd.to_datetime(orders_df['order date (DateOrders)'], errors='coerce')

VAL_FRACTION = 0.12
TEST_FRACTION = 0.18

orders_sorted = orders_df.sort_values('order date (DateOrders)').reset_index(drop=True)
n = len(orders_sorted)

test_start = int(n * (1 - TEST_FRACTION))
val_start = int(n * (1 - TEST_FRACTION - VAL_FRACTION))

train_df = orders_sorted.iloc[:val_start]
val_df = orders_sorted.iloc[val_start:test_start]
test_df = orders_sorted.iloc[test_start:]

for name, part in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    print(f"{name}: {len(part):,} orders ({len(part)/n*100:.1f}%), "
          f"{part['order date (DateOrders)'].min()} to {part['order date (DateOrders)'].max()}")


Train: 46,026 orders (70.0%), 2015-01-01 00:00:00 to 2017-03-16 19:22:00
Validation: 7,890 orders (12.0%), 2017-03-16 19:43:00 to 2017-08-02 00:04:00
Test: 11,836 orders (18.0%), 2017-08-02 00:25:00 to 2018-01-31 23:38:00


In [14]:
for name, part in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    pct = part['Late_delivery_risk'].value_counts(normalize=True).round(4) * 100
    print(f"{name}: {pct.to_dict()}")


Train: {1: 54.85, 0: 45.15}
Validation: {1: 54.230000000000004, 0: 45.769999999999996}
Test: {1: 55.14, 0: 44.86}


**What we found:** class balance holds tightly across all three splits (Train 54.85%/45.15%, Validation 54.23%/45.77%, Test 55.14%/44.86%) - max deviation under 1 percentage point from the overall average, confirming EDA's no-drift finding even at this finer granularity. Split sizes: Train 46,026 (70.0%, Jan 2015–Mar 2017), Validation 7,890 (12.0%, Mar–Aug 2017), Test 11,836 (18.0%, Aug 2017–Jan 2018).


In [ ]:
TRAIN_OUTPUT = PROCESSED_DIR / 'train.csv'
VAL_OUTPUT = PROCESSED_DIR / 'val.csv'
TEST_OUTPUT = PROCESSED_DIR / 'test.csv'

train_df.to_csv(TRAIN_OUTPUT, index=False)
val_df.to_csv(VAL_OUTPUT, index=False)
test_df.to_csv(TEST_OUTPUT, index=False)

print(f"Saved: {TRAIN_OUTPUT.name}, {VAL_OUTPUT.name}, {TEST_OUTPUT.name} to {PROCESSED_DIR.resolve()}")
print(f"Final feature set: {list(orders_df.columns)}")


## Summary 

| # | Decision | Reasoning |
|---|---|---|
| 1 | Aggregation: mode (not `first`) for `Category Name`/`Category Id`; added `n_line_items`, `n_distinct_categories` | 67.77% of orders span multiple categories - `first` would have silently discarded this for most of the dataset |
| 2 | `Order Zipcode` dropped entirely | 87.42% missing, confirmed systemic (0% availability across most countries) - a flag would be redundant with `Order Country` |
| 3 | No `Order Country` standardization needed | EDA's "English/Spanish mixing" claim was based on an incomplete sample; the full list is consistently Spanish |
| 4 | `Category Name` confirmed canonical; `Category Id` excluded standalone | Department scoping check confirmed zero inconsistencies once paired with `Department Id` - not a data error, a normal ID scoping convention |
| 5 | `Benefit per order` capped at 1st/99th percentile; `priority_value_component` = `Sales` | Avoids extreme values dominating training and avoids negative profit orders being miscounted in the secondary lens's priority ranking |
| 6 | Chronological train (70%) / validation (12%) / test (18%) split | No calendar drift found, but the 2018 tail is too thin to hold out alone; validation split added specifically to keep Optuna tuning computationally feasible |

**Final feature set:** order level categoricals (`Shipping Mode`, `Order Region`, `Order Country`, `Order City`, `Order State`, `Customer Segment`, `Type`, `Category Name`), order level numerics (`Days for shipment (scheduled)`, `Sales`, `Order Item Quantity`, `Benefit_per_order_capped`, `n_line_items`, `n_distinct_categories`), the temporal field (`order date (DateOrders)`), the secondary lens field (`priority_value_component`), and the target (`Late_delivery_risk`).

